# Radar Chart Generator
This notebook creates radar (polar) charts and saves them as PNGs under `notebooks/images`. Use the functions below to build, style, and save charts.

In [ ]:
# Imports
import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

In [ ]:
# Prepare radar chart data
def make_example_data():
    labels = ['Speed','Altitude','Range','Azimuth','Reflectivity','Velocity']
    values = [0.8, 0.6, 0.9, 0.5, 0.7, 0.4]
    return labels, values

In [ ]:
# Radar plot function (single dataset)
def plot_radar(labels, values, ax=None, color='C0', fill_alpha=0.25, title=None, rlim=(0,1)):
    values = np.array(values)
    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False)
    angles = np.concatenate((angles, [angles[0]]))
    values = np.concatenate((values, [values[0]]))
    if ax is None:
        fig, ax = plt.subplots(figsize=(6,6), subplot_kw=dict(polar=True))
    else:
        fig = ax.get_figure()
    ax.set_theta_offset(np.pi/2)
    ax.set_theta_direction(-1)
    ax.plot(angles, values, color=color)
    ax.fill(angles, values, color=color, alpha=fill_alpha)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels)
    ax.set_ylim(*rlim)
    if title:
        ax.set_title(title)
    return fig, ax

In [ ]:
# Radar plot function (multiple datasets)
def plot_radar_multiple(labels, dataset_list, colors=None, labels_legend=None, rlim=(0,1), title=None):
    n = len(dataset_list)
    if colors is None:
        colors = plt.cm.tab10.colors
    fig, axes = plt.subplots(1,1, subplot_kw=dict(polar=True), figsize=(6,6))
    ax = axes
    for i, values in enumerate(dataset_list):
        c = colors[i % len(colors)]
        plot_radar(labels, values, ax=ax, color=c, fill_alpha=0.15, rlim=rlim)
    if labels_legend:
        ax.legend(labels_legend, loc='upper right', bbox_to_anchor=(1.1, 1.1))
    if title:
        ax.set_title(title)
    return fig, ax

In [ ]:
# Styling, annotations, and labels
def style_example(ax):
    ax.grid(color='gray', linestyle='--', alpha=0.5)
    for label in ax.get_xticklabels():
        label.set_fontsize(10)
    ax.tick_params(axis='y', labelsize=8)

In [ ]:
# Save chart as PNG in notebooks/images
def save_figure(fig, out_dir='notebooks/images', fname=None, dpi=150, transparent=False):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    if fname is None:
        fname = f"radar_{datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}.png"
    path = out / fname
    fig.savefig(path, bbox_inches='tight', dpi=dpi, transparent=transparent)
    return path

In [ ]:
# Display saved PNG inline
labels, values = make_example_data()
fig, ax = plot_radar(labels, values, title='Example Radar')
style_example(ax)
path = save_figure(fig)
display(Image(filename=str(path)))

In [ ]:
# Simple unit test to verify PNG creation
def test_save_png(tmp_path):
    labels, values = make_example_data()
    fig, ax = plot_radar(labels, values)
    out = tmp_path / 'test_images'
    out.mkdir()
    p = out / 'test.png'
    fig.savefig(p, bbox_inches='tight')
    assert p.exists() and p.stat().st_size > 0